In [6]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy import stats

In [7]:
df = pd.read_csv(r'full_data_2.csv')
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 991 entries, 0 to 990
Data columns (total 46 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Unnamed: 0         991 non-null    int64  
 1   Name               991 non-null    str    
 2   Price              350 non-null    float64
 3   Link               991 non-null    str    
 4   Screen Size        886 non-null    float64
 5   Display            991 non-null    int64  
 6   Rear Camera        868 non-null    str    
 7   Front Camera       838 non-null    str    
 8   Chipset            991 non-null    int64  
 9   NFC                991 non-null    int64  
 10  ROM                938 non-null    float64
 11  SIM Card           713 non-null    str    
 12  Operating System   774 non-null    str    
 13  Screen Resolution  678 non-null    str    
 14  Display Features   745 non-null    str    
 15  CPU                605 non-null    str    
 16  RAM                916 non-null    fl

In [8]:
def compute_correlation(df, target, method: str = "pearson", min_samples: int = 30, exclude_cols: list = None):
    """
    Tính hệ số tương quan giữa tất cả biến số với một biến mục tiêu.
 
    Tham số
    -------
    df          : DataFrame chứa dữ liệu
    target      : tên cột mục tiêu (mặc định 'antutu_11')
    method      : 'pearson', 'spearman', hoặc 'kendall'
    min_samples : số mẫu tối thiểu (sau khi bỏ NaN) để tính tương quan
    exclude_cols: danh sách cột muốn bỏ qua (ngoài target)
 
    Trả về
    ------
    DataFrame gồm: feature, correlation, p_value, n_samples, strength
    Sắp xếp theo |correlation| giảm dần.
    """
    if target not in df.columns:
        raise ValueError(f"Cột '{target}' không tồn tại trong DataFrame.")
 
    exclude = set(exclude_cols or []) | {target}
    numeric_cols = [c for c in df.select_dtypes(include="number").columns
                    if c not in exclude]
 
    target_series = df[target]
    records = []
 
    for col in numeric_cols:
        pair = df[[col, target]].dropna()
        n = len(pair)
        if n < min_samples:
            continue
 
        x, y = pair[col].values, pair[target].values
 
        if method == "pearson":
            r, p = stats.pearsonr(x, y)
        elif method == "spearman":
            r, p = stats.spearmanr(x, y)
        elif method == "kendall":
            r, p = stats.kendalltau(x, y)
        else:
            raise ValueError("method phải là 'pearson', 'spearman', hoặc 'kendall'")
 
        abs_r = abs(r)
        if   abs_r >= 0.7: strength = "rất mạnh"
        elif abs_r >= 0.5: strength = "mạnh"
        elif abs_r >= 0.3: strength = "trung bình"
        elif abs_r >= 0.1: strength = "yếu"
        else:              strength = "rất yếu"
 
        records.append({
            "feature":     col,
            "correlation": round(r, 4),
            "p_value":     round(p, 6),
            "n_samples":   n,
            "strength":    strength,
        })
 
    result = (pd.DataFrame(records)
                .assign(abs_corr=lambda d: d["correlation"].abs())
                .sort_values("abs_corr", ascending=False)
                .drop(columns="abs_corr")
                .reset_index(drop=True))
    return result

In [9]:
result = compute_correlation(df, 'antutu_11')

In [10]:
result

,feature,correlation,p_value,n_samples,strength
0,max_freq_ghz,0.9204,0.000000,93,rất mạnh
1,clock,0.9181,0.000000,764,rất mạnh
2,Price,0.7543,0.000000,264,rất mạnh
3,Camera_score,0.6732,0.000000,151,mạnh
4,rear_telephoto,0.6117,0.000000,656,mạnh
5,RAM,0.5743,0.000000,707,mạnh
6,ROM,0.5273,0.000000,717,mạnh
7,PPI,0.4919,0.000000,764,trung bình
8,SIM_total,0.4893,0.000000,764,trung bình
9,perf_freq_ghz,0.4790,0.000000,764,trung bình
